In [ ]:
import pandas as pd
from pathlib import Path
import os

os.chdir('/data2/zhoulab/fanweiliang/mouse_IER1_splicing')
design = pd.read_table('config/samples.mouse_muscle.tsv')

common_columns = {'chr': 'string', 'start': 'Int64', 'end': 'Int64', 'strand': 'Int64', 'intron_motif': 'Int64', 'annotated': 'Int64'}
data_columns = {'unique_reads': 'Int64', 'multi_reads': 'Int64', 'max_overhang': 'Int64'}
junction_file_columns = common_columns | data_columns

all_df = pd.DataFrame(columns=common_columns.keys()).astype(common_columns)

subset = design[design['condition_2'] == '6months']
experiments = {}
for _, row in subset.iterrows():
    experiments.setdefault(row['condition_1'], []).append(row['sample_name'])

for group, samples in experiments.items():
    for n, sample in enumerate(samples):
        idx = n + 1
        normal_df = pd.read_table(
            Path(f'results/mouse_muscle/star_align/{sample}/SJ.out.tab'),
            header=None,
            names=list(junction_file_columns.keys()),
            dtype=junction_file_columns
        )
        normal_df.rename(columns={i: f'{group}.Rep{idx}.{i}' for i in data_columns.keys()}, inplace=True)
        all_df = pd.merge(all_df, normal_df, how='outer', on=list(common_columns.keys()))

all_df.fillna(0, inplace=True)
non_canonical = all_df[all_df['intron_motif'] == 0].copy()


non_canonical['start'] = non_canonical['start'] - 1  # convert 1-based to 0-based

non_canonical.to_csv(f'results/mouse_muscle/junctions/IRE1_KO_vs_WT.non_canonical_junctions.tsv', sep='\t', index=False)


In [ ]:
non_canonical_site_starts = non_canonical[['chr', 'start']].copy().drop_duplicates()
non_canonical_site_ends = non_canonical[['chr', 'end']].copy().drop_duplicates()

non_canonical_site_starts.set_index(['chr', 'start'], inplace=True)
non_canonical_site_ends.set_index(['chr', 'end'], inplace=True)


In [ ]:
nc_df = pd.read_table(f'results/mouse_muscle/junctions/IRE1_KO_vs_WT.non_canonical_junctions.tsv')

for group, samples in experiments.items():
    nc_df[f'{group}_uniq'] = nc_df[[f'{group}.Rep{n+1}.unique_reads' for n in range(len(samples))]].astype(str).agg(','.join, axis=1)
    nc_df[f'{group}_uniq.count'] = nc_df[[f'{group}.Rep{n+1}.unique_reads' for n in range(len(samples))]].sum(axis=1)
    nc_df[f'{group}_multi'] = nc_df[[f'{group}.Rep{n+1}.multi_reads' for n in range(len(samples))]].astype(str).agg(','.join, axis=1)
    nc_df[f'{group}_multi.count'] = nc_df[[f'{group}.Rep{n+1}.multi_reads' for n in range(len(samples))]].sum(axis=1)

# filter by total count >= 6, and at least 2 replicates uniq map not 0
nc_df_filtered = nc_df[(nc_df['WT_uniq.count'] >= 6) & (nc_df['KO_uniq.count'] <= 1)].copy()
condition = (nc_df_filtered[['WT.Rep1.unique_reads', 'WT.Rep2.unique_reads', 'WT.Rep3.unique_reads', 'WT.Rep4.unique_reads']] > 0).sum(axis=1) >= 2
nc_df_filtered = nc_df_filtered[condition]

# drop unused
columns_to_drop = []
for group, samples in experiments.items():
    for n, sample in enumerate(samples):
        idx = n + 1
        columns_to_drop.extend([f'{group}.Rep{idx}.unique_reads', f'{group}.Rep{idx}.multi_reads'])
nc_df_filtered.drop(columns=columns_to_drop, inplace=True)
nc_df_filtered.reset_index(inplace=True, drop=True)

print(len(nc_df_filtered))
nc_df_filtered.head(5)


In [ ]:
import re

# bedops styled bed, 6 columns bed6 + 4 bedops extra columns
vM37_transcripts_attr = pd.read_table(
    'references/mouse_ref/gencode.vM37.transcripts.bed',
    header=None,
    names=['chr', 'start', 'end', 'id', 'score', 'strand', 'source', 'feature', 'frame', 'attr'],
)
vM37_transcripts_attr.set_index('id', drop=False, inplace=True)

def get_transcript_by_site(row):
    overlapping_transcripts_df = vM37_transcripts_attr[
        (vM37_transcripts_attr['chr'] == row['chr']) &
        (vM37_transcripts_attr['start'] <= row['start']) &
        (vM37_transcripts_attr['end'] >= row['end'])
    ]
    if not overlapping_transcripts_df.empty:
        return overlapping_transcripts_df.index.tolist()
    else:
        return None
    
def get_coding_gene_by_transcript(row):
    transcript = vM37_transcripts_attr.loc[row['transcript']]
    # only return protein coding genes
    if 'protein_coding' not in transcript['attr']:
        return None
    symbol = re.sub(r'.*gene_name "(\S*)";.*', r'\1', transcript['attr'])
    return symbol


nc_df_filtered['transcript'] = nc_df_filtered.apply(get_transcript_by_site, axis=1)
nc_df_filtered.dropna(subset=['transcript'], inplace=True) # remove junctions not overlapping with any transcripts
nc_df_filtered['junction_id'] = nc_df_filtered.index.astype(str)
nc_df_filtered = nc_df_filtered.explode('transcript')

nc_df_filtered['genes'] = nc_df_filtered.apply(get_coding_gene_by_transcript, axis=1)
nc_df_filtered.dropna(subset=['genes'], inplace=True) # remove transcripts belongs to non coding genes
nc_df_filtered.reset_index(inplace=True, drop=True)

print(len(nc_df_filtered))
nc_df_filtered.head(5)

In [ ]:
import itertools

# standard bed12
vM37_transcripts = pd.read_table(
    '../references/mouse_ref/gencode.vM37.annotation.bed',
    header=None,
    names=['chr', 'start', 'end', 'id', 'score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 'blockCount', 'blockSizes', 'blockStarts'],
)
vM37_transcripts.set_index('id', drop=False, inplace=True)

def range_diff(r1: list[int], r2: list[int]):
    s1, e1 = r1
    s2, e2 = r2
    endpoints = sorted((s1, s2, e1, e2))
    result = []
    if endpoints[0] == s1 and endpoints[1] != s1:
        result.append((endpoints[0], endpoints[1]))
    if endpoints[3] == e1 and endpoints[2] != e1:
        result.append((endpoints[2], endpoints[3]))
    return result

def range_diff_multi(group1: list[list[int]], r2: list[int]):
    group1_diff = list(itertools.chain(
        *[range_diff(r1, r2) for r1 in group1]
    ))
    return group1_diff

def range_intersect_multi(group1: list[list[int]], r2: list[int]):
    intersects = []
    for r1 in group1:
        if (s := max(r1[0], r2[0])) < (e := min(r1[1], r2[1])):
            intersects.append((s, e))
    return intersects

def get_new_exon_range(row):
    transcript = vM37_transcripts.loc[row['transcript']]
    sj_start = int(row['start']) - int(transcript['start'])
    sj_end = int(row['end']) - int(transcript['start'])
    exon_sizes = list(map(int, transcript['blockSizes'].strip(',').split(',')))
    exon_starts = list(map(int, transcript['blockStarts'].strip(',').split(',')))
    exons = [(int(x), int(x + y)) for x, y in zip(exon_starts, exon_sizes)]

    # assume that non-canonical splicing happens after normal splicing
    # if splice junction start or end locates in intronic region, filter this novel transcript out
    if any(s < sj_start < e for s, e in exons) and any(s < sj_end < e for s, e in exons):
        new_exons = range_diff_multi(exons, [sj_start, sj_end])
        spliced_exons = range_intersect_multi(exons, [sj_start, sj_end])
        return pd.Series({
            'original_exons': exons, 'new_exons': new_exons, 'spliced_exons': spliced_exons
        })
    else:
        normal_sites = list(itertools.chain.from_iterable(exons))
        if (sj_start in normal_sites) and ((row['chr'], sj_start) not in non_canonical_site_ends.index):
            print("sj" + row['junction_id'] + " has normal start in: " + row['transcript'])
        elif (sj_end in normal_sites) and ((row['chr'], sj_end) not in non_canonical_site_starts.index):
            print("sj" + row['junction_id'] + " has normal end in: " + row['transcript'])
        else:
            print("sj" + row['junction_id'] + " is intronic in: " + row['transcript'])
        return pd.Series(data={
            'original_exons': exons, 'new_exons': exons, 'spliced_exons': []
        })

# calculate spliced transcript exons
nc_df_filtered[['original_exons', 'new_exons', 'spliced_exons']] = nc_df_filtered.apply(get_new_exon_range, axis = 1)
nc_df_filtered.to_csv(f'../results/mouse_muscle/{expr}/IRE1_KO_vs_WT.non_canonical_junctions.filtered.tsv', sep='\t', index=False)

In [ ]:
changed_transcripts = nc_df_filtered[nc_df_filtered['spliced_exons'].apply(len) != 0].copy()
# changed_transcripts = nc_df_filtered.copy()
changed_transcripts.reset_index(inplace=True)
changed_transcripts.set_index('transcript', inplace=True)

affected_transcripts_bed = vM37_transcripts.loc[changed_transcripts.index]
spliced_new_bed = affected_transcripts_bed.copy()
intronic_bed = affected_transcripts_bed.copy()

# remove duplicates (one transcript may spliced by multiple junction sites)
affected_transcripts_bed.drop_duplicates(inplace=True)
affected_transcripts_bed.to_csv(f'../results/mouse_muscle/{expr}/spliced_transcripts.original.bed', header=None, sep='\t', index=False)

# generate spliced transcript bed
spliced_new_bed['id'] = spliced_new_bed['id'] + '_junc' + changed_transcripts['junction_id']
def get_new_exon_cols(row):
    exons = row['new_exons']
    block_count = len(exons)
    block_starts = ','.join([str(e[0]) for e in exons]) + ','
    block_sizes = ','.join([str(e[1] - e[0]) for e in exons]) + ','
    return pd.Series({
        'blockCount': block_count, 'blockSizes': block_sizes, 'blockStarts': block_starts
    })

spliced_new_bed[['blockCount', 'blockSizes', 'blockStarts']] = changed_transcripts.apply(get_new_exon_cols, axis=1)
spliced_new_bed.head(10)
spliced_new_bed.to_csv(f'../results/mouse_muscle/{expr}/spliced_transcripts.new.bed', header=None, sep='\t', index=False)

# generate intronic bed
intronic_bed['id'] = intronic_bed['id'] + '_junc' + changed_transcripts['junction_id']
def get_intronic_regions(row):
    spliced_regions = row['spliced_exons']
    block_count = len(spliced_regions)
    block_starts = ','.join([str(e[0]) for e in spliced_regions]) + ','
    block_sizes = ','.join([str(e[1] - e[0]) for e in spliced_regions]) + ','
    return pd.Series({
        'blockCount': block_count, 'blockSizes': block_sizes, 'blockStarts': block_starts
    })

intronic_bed[['blockCount', 'blockSizes', 'blockStarts']] = changed_transcripts.apply(get_intronic_regions, axis=1)

def reset_bed_start_and_end(row):
    start = row['start']
    block_starts = [ int(i) for i in row['blockStarts'].strip(',').split(',') ]
    block_sizes = [ int(i) for i in row['blockSizes'].strip(',').split(',') ]

    new_start = start + block_starts[0]
    new_end = start + block_starts[-1] + block_sizes[-1]
    new_block_starts = [ x - block_starts[0] for x in block_starts ]

    return pd.Series({
        'start': new_start, 'end': new_end, 'thickStart': new_start, 'thickEnd': new_end,
        'blockStarts': ','.join([str(x) for x in new_block_starts]) + ','
    })

intronic_bed[['start', 'end', 'thickStart', 'thickEnd', 'blockStarts']] = intronic_bed.apply(reset_bed_start_and_end, axis=1)
intronic_bed.drop_duplicates(subset=['chr', 'start', 'end', 'strand', 'blockStarts', 'blockSizes'], inplace=True)
intronic_bed.to_csv(f'../results/mouse_muscle/{expr}/spliced_intronic.bed', header=None, sep='\t', index=False)